In [12]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

!pip install -U transformers

# use_colab = False
# if use_colab:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     %cd /content/drive/MyDrive/Final_Project_MLCB

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print("Current Device = ", device)

Current Device =  cpu


In [14]:
main_dir = os.getcwd()
print(main_dir)
data_dir = os.path.join(main_dir, "data/processed")
embedding_dir = os.path.join(main_dir, "data/text_encodings")

c:\Users\nwalr\OneDrive - Georgia Institute of Technology\Grad Semester 2 - Spring 2026\MLCB\MLCB_Project-Hackathon\Final Project


In [15]:
for split in ["train", "val", "test"]:
    df = pd.read_csv(os.path.join(data_dir, f"{split}_dataset_propagated.csv"))
    
    no_desc_mask = df["clean_text"].isna() | (df["clean_text"].str.strip() == "")
    no_desc_ids = df.loc[no_desc_mask, "uniprot_id"].tolist()
    
    out_path = os.path.join(embedding_dir, f"{split}_no_description_ids.txt")
    with open(out_path, "w") as f:
        for uid in no_desc_ids:
            f.write(uid + "\n")
    
    # Sanity check
    total = len(df)
    missing = len(no_desc_ids)
    print(f"{split}: {missing}/{total} proteins missing descriptions "
          f"({100*missing/total:.1f}%) → saved to {out_path}")

train: 13/794 proteins missing descriptions (1.6%) → saved to c:\Users\nwalr\OneDrive - Georgia Institute of Technology\Grad Semester 2 - Spring 2026\MLCB\MLCB_Project-Hackathon\Final Project\data/text_encodings\train_no_description_ids.txt
val: 0/99 proteins missing descriptions (0.0%) → saved to c:\Users\nwalr\OneDrive - Georgia Institute of Technology\Grad Semester 2 - Spring 2026\MLCB\MLCB_Project-Hackathon\Final Project\data/text_encodings\val_no_description_ids.txt
test: 3/97 proteins missing descriptions (3.1%) → saved to c:\Users\nwalr\OneDrive - Georgia Institute of Technology\Grad Semester 2 - Spring 2026\MLCB\MLCB_Project-Hackathon\Final Project\data/text_encodings\test_no_description_ids.txt


## Load Preprocessed-Data & Get Labels

In [16]:
train_data = pd.read_csv(os.path.join(data_dir, "train_dataset_propagated.csv"))
val_data = pd.read_csv(os.path.join(data_dir, "val_dataset_propagated.csv"))
test_data = pd.read_csv(os.path.join(data_dir, "test_dataset_propagated.csv"))

train_data.shape, val_data.shape, test_data.shape

((794, 6), (99, 6), (97, 6))

In [17]:
features = ['uniprot_id', 'go_terms']
train_labels = train_data[features]
val_labels = val_data[features]
test_labels = test_data[features]
train_labels

,uniprot_id,go_terms
0,A0A0C5B5G6,"['GO:0001503', 'GO:0001649', 'GO:0001932', 'GO..."
1,A0JNW5,"['GO:0003674', 'GO:0005215', 'GO:0005319', 'GO..."
2,A0JP26,[]
3,A1A4S6,"['GO:0003674', 'GO:0005096', 'GO:0005575', 'GO..."
4,A1A519,"['GO:0003674', 'GO:0003676', 'GO:0003677', 'GO..."
...,...,...
789,P57105,"['GO:0001936', 'GO:0001937', 'GO:0005575', 'GO..."
790,P57730,"['GO:0001817', 'GO:0001818', 'GO:0002020', 'GO..."
791,P57735,"['GO:0000166', 'GO:0000902', 'GO:0002064', 'GO..."
792,P57737,"['GO:0000139', 'GO:0003674', 'GO:0003779', 'GO..."


## Load Text Embeddings

In [18]:
import glob

# List all .npy files in embedding_dir
npy_files = glob.glob(os.path.join(embedding_dir, "*.npy"))

# Load each .npy file into a dictionary or list
embeddings = {}
for file_path in npy_files:
    key = os.path.basename(file_path).replace('.npy', '')
    embeddings[key] = np.load(file_path)

train_embeddings = embeddings['train_embeddings']
val_embeddings = embeddings['val_embeddings']
test_embeddings = embeddings['test_embeddings']

train_embeddings.shape, val_embeddings.shape, test_embeddings.shape

((794, 768), (99, 768), (97, 768))

In [19]:
class Config:
    # Model parameters
    seed          = 42
    epochs        = 40
    batch_size    = 8
    lr            = 1e-4
    hidden_dim    = 128
    dropout       = 0.5
    weight_decay  = 1e-4
    pos_weight_cap = 10.0
    min_go_freq   = 3
    
    # Data paths
    train_csv = os.path.join(os.getcwd(), "data/processed/train_dataset_propagated.csv")
    val_csv = os.path.join(os.getcwd(), "data/processed/val_dataset_propagated.csv")
    test_csv = os.path.join(os.getcwd(), "data/processed/test_dataset_propagated.csv")
    
    # Embeddings
    train_seq_emb = os.path.join(os.getcwd(), "data/processed/train_embeddings_esm2_embeddings.pkl")
    train_text_emb = os.path.join(os.getcwd(), "data/text_encodings/train_embeddings.npy")

    val_seq_emb = os.path.join(os.getcwd(), "data/processed/val_embeddings_esm2_embeddings.pkl")
    val_text_emb = os.path.join(os.getcwd(), "data/text_encodings/val_embeddings.npy")

    test_seq_emb = os.path.join(os.getcwd(), "data/processed/test_embeddings_esm2_embeddings.pkl")
    test_text_emb = os.path.join(os.getcwd(), "data/text_encodings/test_embeddings.npy")

    no_desc_dir = os.path.join(os.getcwd(), "data/text_encodings")
    
args = Config()

In [20]:
import Sequence_Text as st
from combined_model import GOOntology, information_content, build_label_space, class_pos_weight

print(f"\n{'='*60}\nTraining: TextSeqMLP\n{'='*60}")
torch.manual_seed(args.seed)
np.random.seed(args.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load ontology and compute Information Content (IC) for Smin
ontology = GOOntology("data/go-basic.obo")
ic = information_content(args.train_csv, ontology)

# Build the shared label space (must match final model for comparability)
labels = build_label_space(args.train_csv, args.min_go_freq)
label_to_idx = {t: i for i, t in enumerate(labels)}

print(f"Label space: {len(labels)} GO terms (min_freq={args.min_go_freq})")
if not labels:
    raise SystemExit("No GO terms passed the frequency filter.")

pos_weight = class_pos_weight(args.train_csv, label_to_idx, args.pos_weight_cap).to(device)


Training: TextSeqMLP
Device: cpu
Label space: 3924 GO terms (min_freq=3)


In [ ]:
# import Sequence_Text as st
# from train_baselines import build_label_space, class_pos_weight

# labels       = build_label_space(args.train_csv, args.min_go_freq)
# label_to_idx = {t: i for i, t in enumerate(labels)}
# print(f"Label space: {len(labels)} GO terms (min_freq={args.min_go_freq})")
# if not labels:
#     raise SystemExit("No GO terms passed the frequency filter.")
 
# pos_weight = class_pos_weight(args.train_csv, label_to_idx, cap=args.pos_weight_cap).to(device)

In [21]:
# Train: no fallback — placeholder embedding is the honest input
train_ds = st.SeqTextDataset(args.train_csv, args.train_seq_emb, args.train_text_emb, label_to_idx)
 
# Val / test: load no-description ID sets for fallback
val_ds = st.SeqTextDataset(
        args.val_csv, args.val_seq_emb, args.val_text_emb, label_to_idx,
        no_description_ids=st.load_no_description_ids(
            os.path.join(args.no_desc_dir, "val_no_description_ids.txt"))
    )
test_ds = st.SeqTextDataset(
        args.test_csv, args.test_seq_emb, args.test_text_emb, label_to_idx,
        no_description_ids=st.load_no_description_ids(
            os.path.join(args.no_desc_dir, "test_no_description_ids.txt"))
    )

print(f"Splits — train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")

Dataset Initialized: 794 valid proteins (Dropped 0)
  Loaded 0 no-description IDs from c:\Users\nwalr\OneDrive - Georgia Institute of Technology\Grad Semester 2 - Spring 2026\MLCB\MLCB_Project-Hackathon\Final Project\data/text_encodings\val_no_description_ids.txt
Dataset Initialized: 99 valid proteins (Dropped 0)
  Loaded 3 no-description IDs from c:\Users\nwalr\OneDrive - Georgia Institute of Technology\Grad Semester 2 - Spring 2026\MLCB\MLCB_Project-Hackathon\Final Project\data/text_encodings\test_no_description_ids.txt
Dataset Initialized: 97 valid proteins (Dropped 0)
Splits — train: 794  val: 99  test: 97


In [22]:
import pickle
with open(args.train_seq_emb, "rb") as f:
    seq_data = pickle.load(f)
print(f"Total proteins with sequence embeddings: {len(seq_data)}")
csv_ids = set(train_ds.df["uniprot_id"])
emb_ids = set(train_ds.seq_data.keys())
print(len(csv_ids))
print(len(emb_ids))

missing_in_emb = csv_ids - emb_ids
print(f"Proteins in CSV but missing embeddings: {len(missing_in_emb)}")
print(f"Sample missing IDs: {list(missing_in_emb)[:5]}")

Total proteins with sequence embeddings: 794
794
794
Proteins in CSV but missing embeddings: 0
Sample missing IDs: []


In [23]:
# 1. Identify valid IDs that exist in BOTH the CSV and the Sequence Dictionary
# This prevents KeyErrors for IDs like 'B3KU38'
train_uids = [uid for uid in train_ds.df["uniprot_id"].tolist() if uid in train_ds.seq_data]

# 2. Build the sequence reference matrix using only valid IDs
train_zs = torch.stack([
    torch.tensor(train_ds.seq_data[uid], dtype=torch.float32)
    for uid in train_uids
])
train_zs_norm = F.normalize(train_zs, dim=-1)  # (N_valid, seq_dim)

# 3. Align the text data to match these specific IDs
# We use a mask to ensure the rows in train_zt match the rows in train_zs_norm
valid_mask = train_ds.df["uniprot_id"].isin(train_uids)
train_zt = torch.from_numpy(train_ds.text_data[valid_mask.values]) # (N_valid, text_dim)

print(f"Fallback bank initialized with {len(train_uids)} valid proteins.")

# Now apply fallback to val and test
val_ds = st.apply_inference_fallback(val_ds, train_zs_norm, train_zt)
test_ds = st.apply_inference_fallback(test_ds, train_zs_norm, train_zt)
 
train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=args.batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=args.batch_size, shuffle=False)
 
# Model
seq_dim  = train_ds[0][1].shape[0]
text_dim = train_ds[0][2].shape[0]
in_dim   = seq_dim + text_dim
 
model = st.TextSeqMLP(
        in_dim, len(labels),
        hidden=args.hidden_dim,
        dropout=args.dropout
    ).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
 
print(f"\nModel input dim: {in_dim} (seq={seq_dim}, text={text_dim})")
print(f"Output dim: {len(labels)}")

Fallback bank initialized with 794 valid proteins.
Fallback: no missing descriptions — skipping.
Fallback: substituting text embeddings for 3 proteins...
O75638 → nearest neighbour index 304 (cosine sim=0.947)
P0CG20 → nearest neighbour index 455 (cosine sim=0.976)
P55327 → nearest neighbour index 176 (cosine sim=0.959)

Model input dim: 1408 (seq=640, text=768)
Output dim: 3924


In [26]:
import json
from combined_model import summarize_metrics
 
def train_loop():
    best_val_fmax = -1.0
    best_state    = None

    for ep in range(0, args.epochs):
        train_loss = st.train_epoch(model, train_loader, optimizer, pos_weight, device)
        
        # val_results is now the dictionary returned by the updated evaluate()
        val_results = st.evaluate(model, val_loader, device)
        
        # Use the summarize_metrics harness to get the Fmax for checkpointing
        # Note: You can also use st.fmax_score(val_results["y_true"], val_results["y_prob"])
        val_metrics = summarize_metrics(val_results, labels, ontology, ic)
        current_fmax = val_metrics["overall"]["fmax"]
    
        print(f"ep {ep:3d} | loss={train_loss:.4f} | "
              f"val_fmax={current_fmax:.4f} | val_aupr={val_metrics['overall']['aupr']:.4f}")
    
        if current_fmax > best_val_fmax:
            best_val_fmax = current_fmax
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    # Final evaluation on test set
    test_results = st.evaluate(model, test_loader, device)
    final_metrics = summarize_metrics(test_results, labels, ontology, ic)
    
    print(f"\n{'='*60}")
    print("FINAL TEST METRICS (Namespace Breakdown)")
    print(f"{'='*60}")
    print(json.dumps(final_metrics, indent=2))
    
    return final_metrics

results = train_loop()
results

ep   0 | loss=0.6894 | val_fmax=0.1827 | val_aupr=0.1246
ep   1 | loss=0.4478 | val_fmax=0.3982 | val_aupr=0.3247
ep   2 | loss=0.3899 | val_fmax=0.4089 | val_aupr=0.3493
ep   3 | loss=0.3723 | val_fmax=0.4094 | val_aupr=0.3544
ep   4 | loss=0.3632 | val_fmax=0.4085 | val_aupr=0.3583
ep   5 | loss=0.3576 | val_fmax=0.4090 | val_aupr=0.3591
ep   6 | loss=0.3517 | val_fmax=0.4107 | val_aupr=0.3615
ep   7 | loss=0.3475 | val_fmax=0.4097 | val_aupr=0.3623
ep   8 | loss=0.3462 | val_fmax=0.4101 | val_aupr=0.3631
ep   9 | loss=0.3458 | val_fmax=0.4106 | val_aupr=0.3639
ep  10 | loss=0.3434 | val_fmax=0.4098 | val_aupr=0.3648
ep  11 | loss=0.3404 | val_fmax=0.4109 | val_aupr=0.3646
ep  12 | loss=0.3385 | val_fmax=0.4113 | val_aupr=0.3635
ep  13 | loss=0.3380 | val_fmax=0.4106 | val_aupr=0.3640
ep  14 | loss=0.3364 | val_fmax=0.4114 | val_aupr=0.3639
ep  15 | loss=0.3360 | val_fmax=0.4119 | val_aupr=0.3639
ep  16 | loss=0.3341 | val_fmax=0.4118 | val_aupr=0.3646
ep  17 | loss=0.3334 | val_fmax

{'n': 97,
 'overall': {'fmax': 0.42326702499814145,
  'fmax_threshold': 0.35000000000000003,
  'aupr': 0.3765881241660801,
  'mean_labels_per_protein': 115.63917541503906},
 'MF_BP_CC': {'MF': {'num_terms': 547,
   'fmax': 0.40076412071264633,
   'fmax_threshold': 0.36000000000000004,
   'aupr': 0.3776524086795863,
   'smin': 31.38864864089008,
   'smin_threshold': 0.35000000000000003},
  'BP': {'num_terms': 2923,
   'fmax': 0.3755078862429829,
   'fmax_threshold': 0.32,
   'aupr': 0.32009698271202197,
   'smin': 197.71403557189657,
   'smin_threshold': 0.34},
  'CC': {'num_terms': 449,
   'fmax': 0.5896129297535587,
   'fmax_threshold': 0.38,
   'aupr': 0.5554077661620096,
   'smin': 31.622673315148,
   'smin_threshold': 0.38}},
 'mean_gate_sequence': 0.0,
 'mean_gate_text': 0.0,
 'mean_gate_structure': 0.0}

## Challenges and Limitations

Challenge: The sequence encoder has less proteins that the text encoder which limited how many proteins I used initally.